### XGBoost

In [4]:
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

DATA_DIR = "karate_data_trial"
GESTURES = [
    "jump", "crouch", "move_left", "move_right",
    "high_punch", "low_punch", "strong_kick", "high_kick",
    "hit_combo", "neutral"
]

# Load data
X, y = [], []
for g in GESTURES:
    folder = os.path.join(DATA_DIR, g)
    for f in os.listdir(folder):
        arr = np.load(os.path.join(folder, f))
        X.append(arr.flatten())  # 30*33*3 = 2970
        y.append(g)

X = np.array(X)
y = np.array(y)

# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.4, random_state=42, stratify=y_enc
)

# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train Extra Trees
clf = ExtraTreesClassifier(
    n_estimators=500,
    max_depth=15,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Save model + encoder + scaler
joblib.dump(clf, "karate_extratrees.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(le, "label_encoder.pkl")


Accuracy: 0.3333333333333333
              precision    recall  f1-score   support

      crouch       0.50      1.00      0.67         1
   high_kick       0.00      0.00      0.00         1
  high_punch       0.00      0.00      0.00         1
   hit_combo       0.00      0.00      0.00         2
        jump       0.00      0.00      0.00         1
   low_punch       0.50      1.00      0.67         1
   move_left       0.00      0.00      0.00         2
  move_right       1.00      1.00      1.00         1
     neutral       0.17      1.00      0.29         1
 strong_kick       0.00      0.00      0.00         1

    accuracy                           0.33        12
   macro avg       0.22      0.40      0.26        12
weighted avg       0.18      0.33      0.22        12



c:\Users\Samrudhi\anaconda3\envs\gesturegame\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Samrudhi\anaconda3\envs\gesturegame\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Samrudhi\anaconda3\envs\gesturegame\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

['label_encoder.pkl']

### Live Detection

In [1]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import joblib

clf = joblib.load("karate_extratrees.pkl")
scaler = joblib.load("scaler.pkl")
le = joblib.load("label_encoder.pkl")

SEQUENCE_LENGTH = 30
buffer = deque(maxlen=SEQUENCE_LENGTH)

mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = pose.process(rgb)

    gesture = "neutral"
    if res.pose_landmarks:
        lm_list = [[lm.x, lm.y, lm.z] for lm in res.pose_landmarks.landmark]
        buffer.append(lm_list)

        if len(buffer) == SEQUENCE_LENGTH:
            clip_arr = np.array(buffer).flatten().reshape(1, -1)
            clip_arr = scaler.transform(clip_arr)
            pred = clf.predict(clip_arr)
            gesture = le.inverse_transform(pred)[0]

        mp.solutions.drawing_utils.draw_landmarks(frame, res.pose_landmarks, mp_pose.POSE_CONNECTIONS)

    cv2.putText(frame, gesture, (30,60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)
    cv2.imshow("Karate Gesture Detection", frame)

    key = cv2.waitKey(1)
    if key == 27 or key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
